In [2]:
"""
Proyecto empresa aliada: limpieza y consolidacion de datos de ventas con Pandas.
Hecho por Santiago Gonzalez
"""

from pathlib import Path
import pandas as pd


# 1. Ubicar los archivos de la actividad.
# Carpeta donde se encuentran los cinco archivos
carpeta = Path.home() / "Proyecto_EA3"

# Buscar los archivos en la carpeta
def buscar_archivo(nombre):
    archivo = carpeta / nombre

    if archivo.exists():
        return archivo

    archivo = carpeta / nombre.replace('.', '(1).', 1)

    if archivo.exists():
        return archivo

    raise FileNotFoundError(f"No se encontro {nombre} en {carpeta}")


# 2. Cargar las cinco tablas

df_category = pd.read_csv(buscar_archivo('DIM_CATEGORY.csv'))

df_product = pd.read_excel(
    buscar_archivo('DIM_PRODUCT.xlsx'),
    dtype={'ITEM': 'string'}
)

df_segment = pd.read_excel(
    buscar_archivo('DIM_SEGMENT.xlsx')
)

df_calendar = pd.read_excel(
    buscar_archivo('DIM_CALENDAR.xlsx')
)

df_sales = pd.read_csv(
    buscar_archivo('FACT_SALES.csv'),
    dtype={'ITEM_CODE': 'string'}
)

print("Archivos cargados correctamente")

# 3. Revisar los datos antes de limpiarlos.
tablas = {
    'DIM_CATEGORY': df_category,
    'DIM_PRODUCT': df_product,
    'DIM_SEGMENT': df_segment,
    'DIM_CALENDAR': df_calendar,
    'FACT_SALES': df_sales
}

for nombre, tabla in tablas.items():
    print('\n---', nombre, '---')
    print('Primeras filas:')
    print(tabla.head())
    print('\nInformacion:')
    tabla.info()
    print('\nDescripcion:')
    print(tabla.describe(include='all'))
    print('\nValores nulos por columna:')
    print(tabla.isna().sum())
    print('Filas duplicadas:', tabla.duplicated().sum())


# 4. Limpiar espacios, saltos de linea y formatos de texto.
def limpiar_textos(tabla):
    tabla = tabla.copy()
    for columna in tabla.select_dtypes(include=['object', 'string']).columns:
        tabla[columna] = tabla[columna].astype('string')
        tabla[columna] = tabla[columna].str.replace(r'\s+', ' ', regex=True).str.strip()
        tabla[columna] = tabla[columna].replace('', pd.NA)
        # Normalizar nombres, regiones y caracteristicas; no alterar las claves.
        if columna not in ['ITEM', 'ITEM_CODE', 'WEEK']:
            tabla[columna] = tabla[columna].str.upper()
    return tabla


df_category = limpiar_textos(df_category).drop_duplicates()
df_product = limpiar_textos(df_product).drop_duplicates()
df_segment = limpiar_textos(df_segment).drop_duplicates()
df_calendar = limpiar_textos(df_calendar).drop_duplicates()
df_sales = limpiar_textos(df_sales).drop_duplicates()

# Eliminar filas de productos sin codigo, ya que no se pueden relacionar con ventas.
df_product = df_product.dropna(subset=['ITEM']).drop_duplicates(subset=['ITEM'])

# Conservar solamente claves validas en las tablas de consulta.
df_category = df_category.dropna(subset=['ID_CATEGORY']).drop_duplicates(subset=['ID_CATEGORY'])
df_calendar = df_calendar.dropna(subset=['WEEK']).drop_duplicates(subset=['WEEK'])

claves_segmento = ['CATEGORY', 'ATTR1', 'ATTR2', 'ATTR3', 'FORMAT']
df_segment = df_segment.drop_duplicates(subset=claves_segmento)

# Convertir los valores de venta a numeros, por si existen formatos incorrectos.
columnas_numericas = [
    'TOTAL_UNIT_SALES',
    'TOTAL_VALUE_SALES',
    'TOTAL_UNIT_AVG_WEEKLY_SALES'
]



Archivos cargados correctamente

--- DIM_CATEGORY ---
Primeras filas:
   ID_CATEGORY                        CATEGORY
0            1  FABRIC TREATMENT and SANIT\r\n
1            2                       AIR CARE 
2            3                    LAVAVAJILLAS
3            4                MEGA SUPERFICIES
4            5             LAVATORY CARE & BRC

Informacion:
<class 'pandas.DataFrame'>
RangeIndex: 5 entries, 0 to 4
Data columns (total 2 columns):
 #   Column       Non-Null Count  Dtype
---  ------       --------------  -----
 0   ID_CATEGORY  5 non-null      int64
 1   CATEGORY     5 non-null      str  
dtypes: int64(1), str(1)
memory usage: 296.0 bytes

Descripcion:
        ID_CATEGORY                        CATEGORY
count      5.000000                               5
unique          NaN                               5
top             NaN  FABRIC TREATMENT and SANIT\r\n
freq            NaN                               1
mean       3.000000                             NaN
std     